In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pylab as plt
import cv2
import os
from PIL import Image
from torchvision import transforms
from sklearn.model_selection import train_test_split

In [3]:
path = "labeled_data.csv"
df = pd.read_csv(path)
data = df.copy()
data.head()

,uid,MeSH,findings,impression,filename,label
0,1,normal,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.,1_IM-0001-4001.dcm.png,normal
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.,2_IM-0652-1001.dcm.png,abnormal
2,4,"Pulmonary Disease, Chronic Obstructive;Bullous...",There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...,4_IM-2050-1001.dcm.png,abnormal
3,5,Osteophyte/thoracic vertebrae/multiple/small;T...,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.,5_IM-2117-1003002.dcm.png,abnormal
4,6,normal,Heart size and mediastinal contour are within ...,No acute cardiopulmonary findings.,6_IM-2192-1001.dcm.png,normal


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
filepath = "/content/drive/MyDrive/required_images"

In [6]:
def train_transform_pipeline(filepath):
    transform = transforms.Compose([
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.RandomResizedCrop(size=224, scale=(0.9, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    img = cv2.imread(filepath, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = Image.fromarray(img)
    img = transform(img)
    return img

In [7]:
def transform_pipeline(filepath):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    img = cv2.imread(filepath, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = transform(img)
    return img

In [8]:
import torch

In [9]:
class ChestXrayDataset(torch.utils.data.Dataset):
      def __init__(self,df,filepath,transform_fn):
         self.df = df.reset_index(drop=True)
         self.filepath = filepath
         self.transform_fn = transform_fn

      def __len__(self):
         return len(self.df)

      def __getitem__(self, index):
          img_path = os.path.join(self.filepath,self.df.loc[index,'filename'])
          img = self.transform_fn(img_path)
          label = self.df.loc[index,'label']
          return img, label

In [10]:
train_df,temp_df = train_test_split(
    data,
    test_size = 0.2,
    random_state = 42,
    stratify = data['label']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size = 0.5,
    random_state = 42,
    stratify = temp_df['label']
)

In [11]:
train_dataset = ChestXrayDataset(train_df,filepath,train_transform_pipeline)
val_dataset = ChestXrayDataset(val_df,filepath,transform_pipeline)
test_dataset = ChestXrayDataset(test_df,filepath,transform_pipeline)

In [12]:
from torch.utils.data import DataLoader


In [13]:
train_loader = DataLoader(dataset = train_dataset,batch_size=16,shuffle=True,num_workers=2)
val_loader = DataLoader(dataset = val_dataset,batch_size=16,shuffle=False,num_workers=2)
test_loader = DataLoader(dataset = test_dataset,batch_size=16,shuffle=False,num_workers=2)

In [14]:
for batch_idx,(image,label) in enumerate(train_loader):
    print(f"Batch : {batch_idx} | Features shape : {image.shape} | Labels : {label}")
    break

Batch : 0 | Features shape : torch.Size([16, 3, 224, 224]) | Labels : ('normal', 'abnormal', 'normal', 'normal', 'abnormal', 'abnormal', 'abnormal', 'abnormal', 'normal', 'abnormal', 'abnormal', 'abnormal', 'abnormal', 'abnormal', 'abnormal', 'abnormal')


In [15]:
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights = ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 202MB/s]


In [16]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [17]:
for param in model.parameters():
    param.requires_grad = False

In [18]:
import torch.nn as nn

In [19]:
model.fc = nn.Linear(model.fc.in_features,2)

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [22]:
model = model.to(device)

In [23]:
for batch_idx,(image,label) in enumerate(train_loader):
    image = image.to(device)
    output = model(image)
    break

In [24]:
output.shape

torch.Size([16, 2])